# MOSS Intel iGPU Benchmark

Benchmark only MOSS-Transcribe-Diarize on Intel `xpu:0` (BF16, SDPA). Each trial runs in a fresh process, with fixed CPU threads and cached model weights. Typhoon is not loaded. Optionally enable CPU FP32 trials for comparison.

Select the project `.venv/bin/python` kernel. Install with:

```bash
uv pip install --python .venv -r requirements-research.txt --torch-backend=xpu
```

Before benchmarking, complete one successful run of `research_iGPU.ipynb` to cache weights and remote model code. Workers run offline and report missing cache or unavailable hardware as failures; no CPU fallback is used. Use a short real recording first.

Timing covers the adapter call: loading, audio preprocessing, generation, and cleanup. It excludes process startup, runtime initialization, duration probing, and result saving. These are fresh-process workflow measurements, not warm inference-only latency. RTF is elapsed seconds / audio seconds; lower is faster. No accuracy score is computed without human reference transcripts. XPU allocator metrics exclude driver/context memory and other shared RAM usage.

## Local Intel runtime

On this Ubuntu 24.04 host, run `uv run --no-project --python .venv setup_iGPU.py` once. Select **MOSS Intel iGPU (.venv)** and restart the kernel. This sets the project-local Level Zero loader and driver paths before PyTorch starts; the regular Python kernel does not use them. No system driver is replaced.

Worker limits: 300 seconds per recording/trial and at least 2 GiB available system RAM. A resource limit stops the batch after reaping the worker. The 2,048-token limit bounds generation cache growth; reaching it is reported as failure, not a complete transcript. Split long recordings before raising limits. Progress prints every 128 generated tokens.


In [ ]:
import sys
from pathlib import Path
import torch

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "speech_to_text").is_dir():
    raise RuntimeError("Open this notebook from the project root.")
if Path(sys.prefix).resolve() != (PROJECT_ROOT / ".venv").resolve():
    raise RuntimeError("Select the project .venv kernel.")
print("Python:", sys.executable)
print("PyTorch:", torch.__version__)
print("XPU available:", torch.xpu.is_available())
print("XPU device count:", torch.xpu.device_count())
if torch.xpu.is_available():
    print("GPU:", torch.xpu.get_device_name(0))
else:
    print("XPU trials will fail until the Intel driver/runtime detects the GPU.")

## Inputs and configuration

All supported recordings under `audio/active/` are processed sequentially. Repeats load and unload the model independently. `INCLUDE_CPU` adds potentially slow CPU FP32 trials. Keep language hints, audio, token limits, and CPU threads identical when comparing devices.

In [ ]:
AUDIO_DIR = PROJECT_ROOT / "audio" / "active"
EXTENSIONS = {".wav", ".mp3", ".flac", ".m4a", ".ogg", ".aac", ".opus", ".wma"}
audio_files = sorted(
    p for p in AUDIO_DIR.rglob("*")
    if p.is_file() and not p.is_symlink() and p.suffix.lower() in EXTENSIONS
    and p.resolve().is_relative_to(AUDIO_DIR.resolve())
)
REPEATS = 3
CPU_THREADS = 4
MAX_NEW_TOKENS = 2048
WORKER_TIMEOUT_SECONDS = 300
MIN_AVAILABLE_RAM_GIB = 2
LANGUAGES = ["Thai", "English"]
INCLUDE_CPU = False
for p in audio_files:
    print(p.relative_to(PROJECT_ROOT))
if not audio_files:
    raise ValueError("Add real recordings to audio/active before benchmarking.")

## Run isolated trials

Only one model worker runs at a time. The next trial starts only after the previous worker exits, releasing its RAM and shared iGPU memory. The notebook kernel does not load model weights.

Each trial saves its request, result metadata, and successful transcript. The adapter uses shared cleanup in `finally`; XPU post-cleanup allocation must return to the baseline. Process exit also releases its GPU context. Interrupting the notebook terminates and waits for the active worker (force-killing it if necessary) before returning control. Inspect completed trial files before starting a new batch.


In [ ]:
import tempfile
from speech_to_text.benchmark import run_benchmarks

output_root = PROJECT_ROOT / "outputs"
output_root.mkdir(exist_ok=True)
benchmark_dir = Path(tempfile.mkdtemp(prefix="benchmark-iGPU-", dir=output_root))
cases = [("moss", "xpu:0")]
if INCLUDE_CPU:
    cases.insert(0, ("moss", "cpu"))
rows = run_benchmarks(
    audio_files, benchmark_dir, cpu_threads=CPU_THREADS, repeats=REPEATS,
    moss_tokens=MAX_NEW_TOKENS, languages=LANGUAGES, cases=cases,
    timeout_seconds=WORKER_TIMEOUT_SECONDS,
    min_available_memory_bytes=MIN_AVAILABLE_RAM_GIB * 1024**3,
)
print("Results:", benchmark_dir)
print("Failures:", sum(row["status"] != "success" for row in rows))

## Summarize results

Only successful trials contribute to medians. Failures are listed explicitly. Speedup uses median CPU FP32 workflow time divided by median XPU BF16 workflow time, and therefore also reflects precision differences. Peak memory is the largest observed tensor allocation across successful trials.

In [ ]:
import json
from collections import defaultdict
from statistics import median
from html import escape
from IPython.display import HTML, display

groups = defaultdict(list)
for row in rows:
    groups[(row["source"], row["device"])].append(row)
summary = []
for (source, device), trials in groups.items():
    passed = [r for r in trials if r["status"] == "success"]
    summary.append({
        "source": source, "device": device,
        "successes": len(passed), "failures": len(trials) - len(passed),
        "median_seconds": median(r["elapsed_seconds"] for r in passed) if passed else None,
        "median_rtf": median(r["rtf"] for r in passed if r.get("rtf") is not None)
            if any(r.get("rtf") is not None for r in passed) else None,
        "peak_allocated_bytes": max((r.get("peak_allocated_bytes", 0) for r in passed), default=None),
    })
for item in summary:
    cpu = next((r for r in summary if r["source"] == item["source"] and r["device"] == "cpu"), None)
    item["cpu_over_xpu_speedup"] = (
        cpu["median_seconds"] / item["median_seconds"]
        if item["device"].startswith("xpu") and cpu and cpu["median_seconds"]
        and item["median_seconds"] else None
    )
columns = list(summary[0]) if summary else []
report = "<h1>MOSS Intel iGPU benchmark</h1><p>Fresh-process workflow timings; CPU FP32 versus XPU BF16.</p>"
report += "<table border='1'><tr>" + "".join(f"<th>{escape(c)}</th>" for c in columns) + "</tr>"
for item in summary:
    report += "<tr>" + "".join(f"<td>{escape(str(item[c]))}</td>" for c in columns) + "</tr>"
report += "</table><h2>Failures</h2><ul>"
for row in rows:
    if row["status"] != "success":
        report += f"<li>{escape(row['source'])}: {escape(row['device'])}: {escape(row.get('error', 'Worker failed'))}</li>"
report += "</ul>"
(benchmark_dir / "summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
(benchmark_dir / "summary.html").write_text(
    '<!doctype html><html lang="en"><meta charset="utf-8"><title>MOSS iGPU benchmark</title><body>' + report + '</body></html>',
    encoding="utf-8",
)
display(HTML(report))
print("Saved results.json, results.csv, summary.json, and summary.html in", benchmark_dir)
if any(row["status"] != "success" for row in rows):
    print("Benchmark incomplete: failed trials do not establish inference performance.")